# Reversible Circuits Project: Phase 5

Simplification that uses two facts the transformation rules cannot use: ancilla lines start at 0, and garbage outputs are never read.

- **Rules and optimizer:** Feng and Li, *A Complete Set of Transformation Rules for Reversible Circuits*, IEEE TCAD, 2026.
- **Adder design:** Kumar et al., *An Optimised Design of a 4-bit Carry Look-Ahead Adder Using Novel Reversible Logic Gates*, IACIS 2024.

**How to run:** choose **Runtime > Run all** from the menu. The whole notebook takes under a minute.

| Step | What it does |
|---|---|
| 1 | Creates the earlier files (unchanged) |
| 2 | Creates `simplify.py`, the ancilla- and garbage-aware simplifier |
| 3 | Creates `verify_phase5.py`, the Phase 5 tests and experiments |
| 4 | Runs them |
| 5 | Examples you can edit and re-run |

**What "equivalent" means in this phase.** Two circuits are design-equivalent when they give the same primary outputs for every value of the primary inputs, with all ancilla lines starting at 0. This is weaker than the equivalence of the paper's rules, which covers every line and every input. The steps here are heuristics of this project; the paper leaves the case with ancillary bits open.

**Reminder.** The carry look-ahead netlist is a reconstruction from the adder paper's block diagram (see Phase 4).

## Step 1: Earlier files (unchanged)

In [ ]:
%%writefile revcircuit.py
"""Reversible circuit model and simulator (Phase 1).

Gates are mixed-polarity multiple-control Toffoli (MPMCT) gates, written
G[P, N, q] as in Feng and Li, "A Complete Set of Transformation Rules for
Reversible Circuits", IEEE TCAD, 2026:

    P  set of positive control lines
    N  set of negative control lines
    q  target line

The gate flips q when every line in P is 1 and every line in N is 0.
Special cases: X[q] = G[{}, {}, q], CNOT[p, q] = G[{p}, {}, q], and a
Toffoli gate is G[{a, b}, {}, q].

A basis state is stored as an integer whose bit i is the value of line q_i.
A circuit is a list of gates; the leftmost gate executes first.
"""
from __future__ import annotations

import random
from dataclasses import dataclass
from itertools import product
from typing import Iterable, Iterator, List, Optional, Sequence, Tuple


# ---------------------------------------------------------------------------
# Gate
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class Gate:
    """One MPMCT gate G[pos, neg, target]."""

    pos: frozenset
    neg: frozenset
    target: int

    def __post_init__(self) -> None:
        object.__setattr__(self, "pos", frozenset(self.pos))
        object.__setattr__(self, "neg", frozenset(self.neg))
        if self.pos & self.neg:
            raise ValueError("a line cannot be both a positive and a negative control")
        if self.target in self.pos or self.target in self.neg:
            raise ValueError("the target line cannot also be a control")
        if min(self.lines) < 0:
            raise ValueError("line numbers must be non-negative")
        # Bit masks used by the simulator.
        object.__setattr__(self, "_pmask", sum(1 << l for l in self.pos))
        object.__setattr__(self, "_nmask", sum(1 << l for l in self.neg))
        object.__setattr__(self, "_tmask", 1 << self.target)

    @property
    def controls(self) -> frozenset:
        return self.pos | self.neg

    @property
    def lines(self) -> frozenset:
        return self.pos | self.neg | {self.target}

    def apply(self, state: int) -> int:
        """Return the state after this gate acts on `state`."""
        if (state & self._pmask) == self._pmask and not (state & self._nmask):
            return state ^ self._tmask
        return state

    def __str__(self) -> str:
        t = self.target
        if not self.neg:
            if not self.pos:
                return f"X[q{t}]"
            if len(self.pos) == 1:
                (c,) = self.pos
                return f"CNOT[q{c},q{t}]"

        def fmt(s: frozenset) -> str:
            return "{" + ",".join(f"q{l}" for l in sorted(s)) + "}"

        return f"G[{fmt(self.pos)},{fmt(self.neg)},q{t}]"


def G(pos: Iterable[int], neg: Iterable[int], target: int) -> Gate:
    """General gate G[P, N, q]."""
    return Gate(frozenset(pos), frozenset(neg), target)


def X(q: int) -> Gate:
    """NOT gate on line q."""
    return Gate(frozenset(), frozenset(), q)


def CNOT(control: int, target: int) -> Gate:
    """Controlled-NOT (Feynman gate)."""
    return Gate(frozenset({control}), frozenset(), target)


def TOF(c1: int, c2: int, target: int) -> Gate:
    """Toffoli gate with two positive controls."""
    return Gate(frozenset({c1, c2}), frozenset(), target)


def MCT(controls: Iterable[int], target: int) -> Gate:
    """Multiple-control Toffoli gate with positive controls only."""
    return Gate(frozenset(controls), frozenset(), target)


# ---------------------------------------------------------------------------
# Circuit
# ---------------------------------------------------------------------------
class Circuit:
    """A reversible circuit on n lines: an ordered list of gates."""

    def __init__(self, n: int, gates: Iterable[Gate] = ()) -> None:
        self.n = n
        self.gates: List[Gate] = list(gates)
        for g in self.gates:
            if max(g.lines) >= n:
                raise ValueError(f"gate {g} uses a line outside q0..q{n - 1}")

    # -- container behaviour ------------------------------------------------
    def __len__(self) -> int:
        return len(self.gates)

    def __getitem__(self, i: int) -> Gate:
        return self.gates[i]

    def __iter__(self) -> Iterator[Gate]:
        return iter(self.gates)

    def __eq__(self, other: object) -> bool:
        """Structural equality: same lines and the same gate list."""
        return isinstance(other, Circuit) and self.n == other.n and self.gates == other.gates

    def __add__(self, other: "Circuit") -> "Circuit":
        """Concatenation: self executes first, then other."""
        if self.n != other.n:
            raise ValueError("circuits must have the same number of lines")
        return Circuit(self.n, self.gates + other.gates)

    def __str__(self) -> str:
        return " ".join(str(g) for g in self.gates) if self.gates else "(empty)"

    def copy(self) -> "Circuit":
        return Circuit(self.n, self.gates)

    def replace(self, i: int, j: int, new_gates: Sequence[Gate]) -> "Circuit":
        """Return a new circuit with gates[i:j] replaced by new_gates."""
        return Circuit(self.n, self.gates[:i] + list(new_gates) + self.gates[j:])

    def inverse(self) -> "Circuit":
        """Every gate is its own inverse, so the inverse is the reversed list."""
        return Circuit(self.n, reversed(self.gates))

    # -- simulation ---------------------------------------------------------
    def apply(self, state: int) -> int:
        for g in self.gates:
            state = g.apply(state)
        return state

    def permutation(self) -> Tuple[int, ...]:
        """The reversible function computed, as a tuple over all 2^n states."""
        return tuple(self.apply(s) for s in range(1 << self.n))

    def equivalent(self, other: "Circuit") -> bool:
        """Functional equivalence: both circuits compute the same function."""
        return self.n == other.n and self.permutation() == other.permutation()

    # -- display ------------------------------------------------------------
    def _bits(self, state: int) -> str:
        return " ".join(str((state >> i) & 1) for i in range(self.n))

    def truth_table(self) -> str:
        head = " ".join(f"q{i}" for i in range(self.n))
        rows = [f"{head} | {head}"]
        for s, t in enumerate(self.permutation()):
            pad = "  ".join(self._bits(s).split())
            out = "  ".join(self._bits(t).split())
            rows.append(f" {pad} |  {out}")
        return "\n".join(rows)

    def draw(self) -> str:
        """Text drawing: circled plus = target, filled dot = positive control, open dot = negative control."""
        width = len(f"q{self.n - 1}")
        rows = []
        for r in range(self.n):
            cells = []
            for g in self.gates:
                lo, hi = min(g.lines), max(g.lines)
                if r == g.target:
                    ch = "\u2295"  # target
                elif r in g.pos:
                    ch = "\u25cf"  # filled dot
                elif r in g.neg:
                    ch = "\u25cb"  # open dot
                elif lo < r < hi:
                    ch = "\u253c"  # wire crossing a gate
                else:
                    ch = "\u2500"
                cells.append("\u2500" + ch + "\u2500")
            rows.append(f"q{r}:".ljust(width + 2) + "\u2500" + "".join(cells) + "\u2500")
        return "\n".join(rows)


# ---------------------------------------------------------------------------
# Helpers for experiments
# ---------------------------------------------------------------------------
def pn_choices(lines: Sequence[int]) -> Iterator[Tuple[frozenset, frozenset]]:
    """All ways to make each given line absent, a positive or a negative control."""
    for states in product((0, 1, 2), repeat=len(lines)):
        pos = frozenset(l for l, s in zip(lines, states) if s == 1)
        neg = frozenset(l for l, s in zip(lines, states) if s == 2)
        yield pos, neg


def all_gates(n: int) -> List[Gate]:
    """Every MPMCT gate on n lines: n * 3^(n-1) gates."""
    gates = []
    for target in range(n):
        others = [l for l in range(n) if l != target]
        for pos, neg in pn_choices(others):
            gates.append(Gate(pos, neg, target))
    return gates


def random_gate(n: int, rng: Optional[random.Random] = None) -> Gate:
    rng = rng or random
    target = rng.randrange(n)
    pos, neg = set(), set()
    for l in range(n):
        if l == target:
            continue
        s = rng.randrange(3)
        if s == 1:
            pos.add(l)
        elif s == 2:
            neg.add(l)
    return Gate(frozenset(pos), frozenset(neg), target)


def random_circuit(n: int, length: int, rng: Optional[random.Random] = None) -> Circuit:
    return Circuit(n, [random_gate(n, rng) for _ in range(length)])

In [ ]:
%%writefile rules.py
"""Transformation rules for reversible circuits (Phase 1).

Rules 1 to 5 are the complete rule set RC of Feng and Li (IEEE TCAD, 2026).
Rules 6 to 10 are the rules the paper derives from RC.

Every rewrite function takes a circuit and a position i and returns a NEW
circuit with the rule applied at position i, or None when the rule does not
match there. The input circuit is never modified.

For rules stated on gates rather than positions, a `ruleK_gates` or
`ruleK_sides` helper returns the gate lists on each side of the rule.
"""
from __future__ import annotations

from itertools import combinations
from typing import Iterable, List, Optional, Sequence, Tuple

from revcircuit import CNOT, Circuit, G, Gate, X


# ===========================================================================
# Rule 1:  A A  ==  (empty)
# ===========================================================================
def rule1_cancel(c: Circuit, i: int) -> Optional[Circuit]:
    """Delete two adjacent identical gates at positions i and i+1."""
    if 0 <= i and i + 1 < len(c) and c[i] == c[i + 1]:
        return c.replace(i, i + 2, [])
    return None


def rule1_insert(c: Circuit, i: int, gate: Gate) -> Circuit:
    """Rule 1 read right to left: insert the pair `gate gate` before position i."""
    return c.replace(i, i, [gate, gate])


# ===========================================================================
# Rule 2:  G[P, N+{p}, q] G[P+{p}, N, q]  ==  G[P, N, q]
# ===========================================================================
def rule2_gates(gate: Gate, p: int) -> List[Gate]:
    """The pair A0, A1 that Rule 2 merges into `gate`, using extra line p."""
    if p in gate.lines:
        raise ValueError("p must be a line the gate does not use")
    return [G(gate.pos, gate.neg | {p}, gate.target), G(gate.pos | {p}, gate.neg, gate.target)]


def rule2_merge(c: Circuit, i: int) -> Optional[Circuit]:
    """Merge gates i and i+1 when they are A0 A1 in the order of the paper."""
    if i < 0 or i + 1 >= len(c):
        return None
    a0, a1 = c[i], c[i + 1]
    extra = a1.pos - a0.pos
    if a0.target != a1.target or len(extra) != 1:
        return None
    (p,) = extra
    merged = G(a1.pos - {p}, a1.neg, a1.target)
    if rule2_gates(merged, p) == [a0, a1]:
        return c.replace(i, i + 2, [merged])
    return None


def rule2_split(c: Circuit, i: int, p: int) -> Optional[Circuit]:
    """Rule 2 read right to left: split gate i into A0 A1 on unused line p."""
    if not (0 <= i < len(c)) or not (0 <= p < c.n) or p in c[i].lines:
        return None
    return c.replace(i, i + 1, rule2_gates(c[i], p))


# ===========================================================================
# Rule 3:  A B == B A  when a control has opposite polarity in A and B
# ===========================================================================
def rule3_condition(a: Gate, b: Gate) -> bool:
    return bool(a.pos & b.neg) or bool(b.pos & a.neg)


def rule3_commute(c: Circuit, i: int) -> Optional[Circuit]:
    """Swap gates i and i+1 when they share a control of opposite polarity."""
    if 0 <= i and i + 1 < len(c) and rule3_condition(c[i], c[i + 1]):
        return c.replace(i, i + 2, [c[i + 1], c[i]])
    return None


# ===========================================================================
# Rule 4:  A B A C1 == C2 A B A,  A = CNOT[p,q], B = CNOT[q,p]
#          (moving a swap of p and q through a gate)
# ===========================================================================
def rule4_sides(p: int, q: int, P: Iterable[int], N: Iterable[int],
                negative: bool = False) -> Tuple[List[Gate], List[Gate]]:
    """Both sides of Rule 4.

    negative=False is condition (a): C1 has q as positive control, target p.
    negative=True  is condition (b): C1 has q as negative control, target p.
    """
    P, N = frozenset(P), frozenset(N)
    a, b = CNOT(p, q), CNOT(q, p)
    if negative:
        c1, c2 = G(P, N | {q}, p), G(P, N | {p}, q)
    else:
        c1, c2 = G(P | {q}, N, p), G(P | {p}, N, q)
    return [a, b, a, c1], [c2, a, b, a]


def _is_cnot(g: Gate) -> bool:
    return len(g.pos) == 1 and not g.neg


def rule4_apply(c: Circuit, i: int) -> Optional[Circuit]:
    """Apply Rule 4 to the four gates starting at i, in either direction."""
    if i < 0 or i + 4 > len(c):
        return None
    seg = c.gates[i:i + 4]
    # Left to right:  A B A C1  ->  C2 A B A
    if _is_cnot(seg[0]):
        (p,) = seg[0].pos
        q = seg[0].target
        c1 = seg[3]
        if c1.target == p and q in c1.controls:
            lhs, rhs = rule4_sides(p, q, c1.pos - {q}, c1.neg - {q}, q in c1.neg)
            if seg == lhs:
                return c.replace(i, i + 4, rhs)
    # Right to left:  C2 A B A  ->  A B A C1
    if _is_cnot(seg[1]):
        (p,) = seg[1].pos
        q = seg[1].target
        c2 = seg[0]
        if c2.target == q and p in c2.controls:
            lhs, rhs = rule4_sides(p, q, c2.pos - {p}, c2.neg - {p}, p in c2.neg)
            if seg == rhs:
                return c.replace(i, i + 4, lhs)
    return None


# ===========================================================================
# Rule 5:  A0 A1 B1..Bm..B1 A1 A0  ==  B'1..B'm..B'1
#          (changes the polarity of control q0)
# ===========================================================================
def rule5_sides(P: Iterable[int], N: Iterable[int], Q: Sequence[int],
                q0: int) -> Tuple[List[Gate], List[Gate]]:
    """Both sides of Rule 5 for control sets P, N, ordered list Q and line q0."""
    P, N, Q = frozenset(P), frozenset(N), list(Q)
    Qs = frozenset(Q)
    if not Q or len(Qs) != len(Q):
        raise ValueError("Q must be a non-empty list of distinct lines")
    a0 = G(P, N | Qs, q0)
    a1 = G(P | Qs, N, q0)
    b = [G(P | {q0} | frozenset(Q[k + 1:]), N | frozenset(Q[:k]), Q[k]) for k in range(len(Q))]
    bp = [G(P | frozenset(Q[k + 1:]), N | {q0} | frozenset(Q[:k]), Q[k]) for k in range(len(Q))]
    lhs = [a0, a1] + b + b[-2::-1] + [a1, a0]
    rhs = bp + bp[-2::-1]
    return lhs, rhs


def rule5_apply(c: Circuit, i: int) -> Optional[Circuit]:
    """Left to right: replace the 2m+3 gate pattern starting at i by 2m-1 gates."""
    if i < 0 or i + 1 >= len(c):
        return None
    a0, a1 = c[i], c[i + 1]
    Qs = a1.pos - a0.pos
    if a0.target != a1.target or not Qs or a0.neg - a1.neg != Qs:
        return None
    if a0.pos - a1.pos or a1.neg - a0.neg:
        return None
    m = len(Qs)
    end = i + 2 * m + 3
    if end > len(c):
        return None
    order = [g.target for g in c.gates[i + 2:i + 2 + m]]
    if frozenset(order) != Qs:
        return None
    lhs, rhs = rule5_sides(a0.pos, a1.neg, order, a0.target)
    if c.gates[i:end] == lhs:
        return c.replace(i, end, rhs)
    return None


def rule5_reverse(c: Circuit, i: int, Q: Sequence[int], q0: int) -> Optional[Circuit]:
    """Right to left: expand B'1..B'm..B'1 at i, given the list Q and line q0."""
    Q = list(Q)
    end = i + 2 * len(Q) - 1
    if i < 0 or not Q or end > len(c):
        return None
    b1 = c[i]
    if b1.target != Q[0] or q0 not in b1.neg:
        return None
    try:
        lhs, rhs = rule5_sides(b1.pos - frozenset(Q[1:]), b1.neg - {q0}, Q, q0)
    except ValueError:
        return None
    if c.gates[i:end] == rhs:
        return c.replace(i, end, lhs)
    return None


# ===========================================================================
# Derived rules 6 to 10
# ===========================================================================
# Rule 6:  G[P,N,q] == product over all subsets S of Q of G[P+S, N+(Q-S), q]
def rule6_gates(gate: Gate, Q: Iterable[int]) -> List[Gate]:
    Q = sorted(set(Q))
    if set(Q) & gate.lines:
        raise ValueError("Q must not overlap the lines of the gate")
    out = []
    for r in range(len(Q) + 1):
        for sub in combinations(Q, r):
            out.append(G(gate.pos | set(sub), gate.neg | (set(Q) - set(sub)), gate.target))
    return out


def rule6_expand(c: Circuit, i: int, Q: Iterable[int]) -> Optional[Circuit]:
    """Expand gate i over all polarity patterns of the unused lines Q."""
    Q = set(Q)
    if not (0 <= i < len(c)) or not Q or Q & c[i].lines or max(Q) >= c.n or min(Q) < 0:
        return None
    return c.replace(i, i + 1, rule6_gates(c[i], Q))


# Rule 7:  two gates with the same target commute
def rule7_commute(c: Circuit, i: int) -> Optional[Circuit]:
    if 0 <= i and i + 1 < len(c) and c[i].target == c[i + 1].target:
        return c.replace(i, i + 2, [c[i + 1], c[i]])
    return None


# Rule 8:  G[P,N,q] == X[N] G[P+N, {}, q] X[N]
def rule8_gates(gate: Gate) -> List[Gate]:
    xs = [X(l) for l in sorted(gate.neg)]
    return xs + [G(gate.pos | gate.neg, (), gate.target)] + xs


def rule8_expand(c: Circuit, i: int) -> Optional[Circuit]:
    """Replace the negative controls of gate i by X gates before and after."""
    if 0 <= i < len(c) and c[i].neg:
        return c.replace(i, i + 1, rule8_gates(c[i]))
    return None


def rule8_collapse(c: Circuit, i: int) -> Optional[Circuit]:
    """Right to left: absorb X..X B X..X starting at i into one gate."""
    k = 0
    while i + k < len(c) and not c[i + k].controls:
        k += 1
    if i < 0 or k == 0 or i + 2 * k + 1 > len(c):
        return None
    b = c[i + k]
    lines = frozenset(g.target for g in c.gates[i:i + k])
    if len(lines) != k or b.neg or not lines <= b.pos:
        return None
    merged = G(b.pos - lines, lines, b.target)
    if c.gates[i:i + 2 * k + 1] == rule8_gates(merged):
        return c.replace(i, i + 2 * k + 1, [merged])
    return None


# Rule 9:  A B1 == B2 A, where A targets p and B1, B2 differ in the polarity of p
def _rule9_pair(a: Gate, b: Gate) -> Optional[Gate]:
    """If A can pass B by Rule 9, return B with the polarity of A's target flipped."""
    p = a.target
    if p not in b.controls:
        return None
    if not (a.pos <= b.pos - {p} and a.neg <= b.neg - {p}):
        return None
    if p in b.pos:
        return G(b.pos - {p}, b.neg | {p}, b.target)
    return G(b.pos | {p}, b.neg - {p}, b.target)


def rule9_apply(c: Circuit, i: int) -> Optional[Circuit]:
    """Move a gate past its neighbour, flipping one control polarity in the neighbour."""
    if i < 0 or i + 1 >= len(c):
        return None
    g1, g2 = c[i], c[i + 1]
    flipped = _rule9_pair(g1, g2)          # A B1 -> B2 A
    if flipped is not None:
        return c.replace(i, i + 2, [flipped, g1])
    flipped = _rule9_pair(g2, g1)          # B2 A -> A B1
    if flipped is not None:
        return c.replace(i, i + 2, [g2, flipped])
    return None


# Rule 10:  A B A == B A B,  A = G[P+{p}, {}, q],  B = G[P+{q}, {}, p]
def rule10_sides(P: Iterable[int], p: int, q: int) -> Tuple[List[Gate], List[Gate]]:
    P = frozenset(P)
    a, b = G(P | {p}, (), q), G(P | {q}, (), p)
    return [a, b, a], [b, a, b]


def rule10_apply(c: Circuit, i: int) -> Optional[Circuit]:
    if i < 0 or i + 3 > len(c):
        return None
    a, b = c[i], c[i + 1]
    p, q = b.target, a.target
    if a.neg or b.neg or p == q or p not in a.pos or q not in b.pos:
        return None
    if a.pos - {p} != b.pos - {q}:
        return None
    lhs, rhs = rule10_sides(a.pos - {p}, p, q)
    if c.gates[i:i + 3] == lhs:
        return c.replace(i, i + 3, rhs)
    return None

In [ ]:
%%writefile optimizer.py
"""Heuristic rule-based optimizer for reversible circuits (Phase 3).

The optimizer rewrites a circuit with the transformation rules of Feng and
Li (IEEE TCAD, 2026) until the cost stops falling. One search step takes a
gate, moves it along the circuit, and looks for a partner it can cancel or
merge with.

Moves that remove gates
    Rule 1          A A                      ->  (nothing)
    Rule 2          G[P,N+p,q] G[P+p,N,q]    ->  G[P,N,q]
    Rule 2 + 1      G[P,N,q]   G[P+p,N,q]    ->  G[P,N+p,q]      ("absorb")
    Rule 5          the 2m+3 gate pattern    ->  2m-1 gates

Moves that carry a gate past its neighbour
    Rule 7          same target: the gates commute
    Rule 3          a shared control of opposite polarity: they commute
    D1 (derived)    neither target is a control of the other: they commute
    Rule 9          the gate passes; a control polarity flips in one gate
    D2 (derived)    the gate passes and leaves one extra gate behind

D1 and D2 are not listed in the paper. Both are sound (verify_phase3.py
checks every instance on 2 to 4 lines), so by the completeness theorem of
the paper they follow from Rules 1 to 5. Rule 9 is the special case of D2
in which the extra gate merges with its neighbour.

Every accepted rewrite strictly lowers the objective, so the search stops.
With check=True each rewrite is also verified by simulating the replaced
gates and the new gates on the lines they use.
"""
from __future__ import annotations

import random
from collections import Counter
from dataclasses import dataclass, field
from typing import Callable, List, Optional, Sequence, Tuple

from revcircuit import Circuit, G, Gate
import rules as R

CostFn = Callable[[Gate], int]


# ---------------------------------------------------------------------------
# Cost model
# ---------------------------------------------------------------------------
def quantum_cost(gate: Gate) -> int:
    """Quantum cost of one mixed-polarity multiple-control Toffoli gate.

    Controls:  0   1   2   3    4    5   ...   c
    Cost:      1   1   5   13   29   61  ...   2^(c+1) - 3

    This is the count of elementary gates in the decomposition of Barenco et
    al. (1995) without ancilla lines. Negative controls are free, except that
    a gate whose controls are ALL negative costs one more. The model is a
    stated assumption of this project; pass another function to change it.
    """
    c = len(gate.pos) + len(gate.neg)
    cost = 1 if c <= 1 else (1 << (c + 1)) - 3
    if c >= 1 and not gate.pos:
        cost += 1
    return cost


def circuit_cost(circuit, cost: CostFn = quantum_cost) -> int:
    """Total quantum cost of a circuit or a list of gates."""
    return sum(cost(g) for g in circuit)


# ---------------------------------------------------------------------------
# Elementary moves
# ---------------------------------------------------------------------------
def reduce_pair(a: Gate, b: Gate) -> Optional[Tuple[List[Gate], str]]:
    """Replace two gates with the same target by at most one gate, if possible."""
    if a.target != b.target:
        return None
    if a == b:
        return [], "Rule 1"
    la, lb = a.controls, b.controls
    if la == lb:
        differ = a.pos ^ b.pos                      # lines with opposite polarity
        if len(differ) == 1:
            (p,) = differ
            return [G(a.pos - {p}, a.neg - {p}, a.target)], "Rule 2"
    elif len(la ^ lb) == 1:
        big, small = (a, b) if len(la) > len(lb) else (b, a)
        (p,) = big.controls - small.controls
        if big.pos - {p} == small.pos and big.neg - {p} == small.neg:
            return [_flip(big, p)], "Rule 2 + 1"
    return None


def _flip(gate: Gate, p: int) -> Gate:
    """The same gate with the polarity of control p reversed."""
    if p in gate.pos:
        return G(gate.pos - {p}, gate.neg | {p}, gate.target)
    return G(gate.pos | {p}, gate.neg - {p}, gate.target)


def commute_tag(a: Gate, b: Gate) -> Optional[str]:
    """Name of the rule by which two adjacent gates commute, or None."""
    if a.target == b.target:
        return "Rule 7"
    if (a.pos & b.neg) or (a.neg & b.pos):
        return "Rule 3"
    if a.target not in b.controls and b.target not in a.controls:
        return "D1"
    return None


def induced_gate(a: Gate, b: Gate) -> Optional[Gate]:
    """The extra gate of rule D2.

    A = G[P1,N1,p] writes line p, and B = G[..p.., q] reads it. Then
        A B  ==  B B'' A      and      B A  ==  A B'' B
    with B'' = G[P1 + P2, N1 + N2, q], where P2, N2 are the controls of B
    other than p. Returns None when the rule does not apply.
    """
    p, q = a.target, b.target
    if p not in b.controls or q in a.controls:
        return None
    pos, neg = a.pos | (b.pos - {p}), a.neg | (b.neg - {p})
    if pos & neg:
        return None                                  # then Rule 3 applies instead
    return G(pos, neg, q)


def pass_gate(moving: Gate, fixed: Gate) -> Optional[Tuple[List[Gate], Gate, str, int]]:
    """Carry `moving` from the left of `fixed` to its right.

    Returns (gates left behind, the moving gate afterwards, rule, extra gates)
    or None when neither Rule 9 nor D2 applies.
    """
    extra = induced_gate(moving, fixed)              # moving writes, fixed reads
    if extra is not None:
        merged = reduce_pair(fixed, extra)
        if merged is not None:
            return merged[0], moving, "Rule 9", 0
        return [fixed, extra], moving, "D2", 1
    extra = induced_gate(fixed, moving)              # fixed writes, moving reads
    if extra is not None:
        merged = reduce_pair(extra, moving)
        if merged is not None:
            return [fixed], merged[0][0], "Rule 9", 0
        return [fixed, extra], moving, "D2", 1
    return None


# ---------------------------------------------------------------------------
# Checking one rewrite
# ---------------------------------------------------------------------------
def same_function(old: Sequence[Gate], new: Sequence[Gate],
                  rng: Optional[random.Random] = None, limit: int = 12) -> bool:
    """True when two gate lists act identically on the lines they use.

    Exhaustive for up to `limit` lines; beyond that, 4096 random states.
    """
    lines = sorted(set().union(*(g.lines for g in list(old) + list(new)))) if (old or new) else []
    index = {l: i for i, l in enumerate(lines)}

    def remap(gates):
        return [G([index[l] for l in g.pos], [index[l] for l in g.neg], index[g.target])
                for g in gates]

    a, b = remap(old), remap(new)
    k = len(lines)
    if k <= limit:
        states = range(1 << k)
    else:
        rng = rng or random.Random(0)
        states = [rng.getrandbits(k) for _ in range(4096)]
    for s in states:
        x = y = s
        for g in a:
            x = g.apply(x)
        for g in b:
            y = g.apply(y)
        if x != y:
            return False
    return True


# ---------------------------------------------------------------------------
# The optimizer
# ---------------------------------------------------------------------------
@dataclass
class OptimizationResult:
    original: Circuit
    circuit: Circuit
    rule_counts: Counter = field(default_factory=Counter)
    rewrites: int = 0
    passes: int = 0
    cost_fn: CostFn = quantum_cost

    @property
    def gates_before(self) -> int:
        return len(self.original)

    @property
    def gates_after(self) -> int:
        return len(self.circuit)

    @property
    def cost_before(self) -> int:
        return circuit_cost(self.original, self.cost_fn)

    @property
    def cost_after(self) -> int:
        return circuit_cost(self.circuit, self.cost_fn)

    def summary(self) -> str:
        def change(before, after):
            pct = 100.0 * (before - after) / before if before else 0.0
            return f"{before} -> {after} ({pct:.1f}% lower)"

        lines = [f"Gate count:   {change(self.gates_before, self.gates_after)}",
                 f"Quantum cost: {change(self.cost_before, self.cost_after)}",
                 f"Rewrites accepted: {self.rewrites} in {self.passes} passes"]
        if self.rule_counts:
            lines.append("Rule usage: " + ", ".join(
                f"{name} x{count}" for name, count in sorted(self.rule_counts.items())))
        return "\n".join(lines)


class _Search:
    def __init__(self, objective: str, window: int, extra_gates: int,
                 cost: CostFn, check: bool) -> None:
        if objective not in ("cost", "gates"):
            raise ValueError('objective must be "cost" or "gates"')
        self.objective, self.window, self.extra_gates = objective, window, extra_gates
        self.cost, self.check = cost, check
        self.counts: Counter = Counter()
        self.rewrites = 0
        self.rng = random.Random(0)

    def score(self, gates: Sequence[Gate]) -> Tuple[int, int]:
        qc, count = sum(self.cost(g) for g in gates), len(gates)
        return (qc, count) if self.objective == "cost" else (count, qc)

    def walk(self, gates: List[Gate], i: int):
        """Move gate i to the right; return the best improving rewrite found."""
        moving, left, tags, growth = gates[i], [], [], 0
        best = None
        for k in range(i + 1, min(len(gates), i + 1 + self.window)):
            fixed = gates[k]
            reduced = reduce_pair(moving, fixed)
            if reduced is not None:
                new = left + reduced[0]
                gain = tuple(o - n for o, n in zip(self.score(gates[i:k + 1]), self.score(new)))
                if gain > (0, 0) and (best is None or gain > best[0]):
                    best = (gain, k, new, tags + [reduced[1]])
            tag = commute_tag(moving, fixed)
            if tag is not None:
                left, tags = left + [fixed], tags + [tag]
                continue
            passed = pass_gate(moving, fixed)
            if passed is None or growth + passed[3] > self.extra_gates:
                break
            left, moving = left + passed[0], passed[1]
            tags, growth = tags + [passed[2]], growth + passed[3]
        return best

    def sweep(self, gates: List[Gate]) -> bool:
        """One left-to-right sweep. Returns True when the circuit changed."""
        changed, i = False, 0
        while i < len(gates):
            best = self.walk(gates, i)
            if best is None:
                i += 1
                continue
            _, k, new, tags = best
            if self.check and not same_function(gates[i:k + 1], new, self.rng):
                raise AssertionError("optimizer produced a non-equivalent rewrite")
            gates[i:k + 1] = new
            self.counts.update(tags)
            self.rewrites += 1
            changed = True
        return changed

    def rule5(self, n: int, gates: List[Gate]) -> bool:
        """Apply Rule 5 wherever its pattern appears literally."""
        changed, i = False, 0
        while i < len(gates):
            new = R.rule5_apply(Circuit(n, gates), i)
            if new is not None and self.score(new.gates) < self.score(gates):
                gates[:] = new.gates
                self.counts["Rule 5"] += 1
                self.rewrites += 1
                changed = True
            else:
                i += 1
        return changed


def optimize(circuit: Circuit, objective: str = "cost", window: int = 50,
             extra_gates: int = 1, cost: CostFn = quantum_cost,
             check: bool = True, max_passes: int = 100) -> OptimizationResult:
    """Reduce a reversible circuit with the transformation rules.

    objective    "cost" minimises quantum cost, then gate count;
                 "gates" minimises gate count, then quantum cost.
    window       how many gates a moving gate may travel past.
    extra_gates  how many extra gates (rule D2) one move may leave behind.
    check        verify every rewrite by simulation on the lines it touches.
    """
    search = _Search(objective, window, extra_gates, cost, check)
    gates = list(circuit.gates)
    passes = 0
    while passes < max_passes:
        passes += 1
        changed = search.sweep(gates)
        # The same search on the reversed circuit moves gates to the left.
        # Each gate is its own inverse, so the reversed circuit is the
        # inverse circuit and every rule stays valid.
        gates.reverse()
        changed |= search.sweep(gates)
        gates.reverse()
        changed |= search.rule5(circuit.n, gates)
        if not changed:
            break
    return OptimizationResult(circuit, Circuit(circuit.n, gates), search.counts,
                              search.rewrites, passes, cost)

In [ ]:
%%writefile bitsim.py
"""Bit-parallel simulation of reversible circuits (Phase 4).

revcircuit.py simulates one input at a time, which is too slow for a
24-line adder (16.7 million inputs). Here every line holds one large
integer whose bit s is the value of that line for input state s, so a gate
is applied to all 2^n inputs at once with a few integer operations.

This makes an exhaustive equivalence check practical up to about 26 lines.
"""
from __future__ import annotations

from typing import List

from revcircuit import Circuit


def input_vectors(n: int) -> List[int]:
    """vectors[i] has bit s equal to bit i of s, for s = 0 .. 2^n - 1."""
    size = 1 << n
    vectors = []
    for i in range(n):
        half = 1 << i
        v = ((1 << half) - 1) << half            # 'half' zeros, then 'half' ones
        length = 2 * half
        while length < size:                     # repeat the pattern to full length
            v |= v << length
            length *= 2
        vectors.append(v)
    return vectors


def output_vectors(circuit: Circuit) -> List[int]:
    """Truth vector of every output line over all 2^n inputs."""
    n = circuit.n
    mask = (1 << (1 << n)) - 1
    lines = input_vectors(n)
    for g in circuit:
        active = mask
        for l in g.pos:
            active &= lines[l]
        for l in g.neg:
            active &= ~lines[l]
        lines[g.target] ^= active & mask
    return lines


def equivalent_all_inputs(a: Circuit, b: Circuit) -> bool:
    """True when the two circuits agree on every line for every one of the 2^n inputs."""
    return a.n == b.n and output_vectors(a) == output_vectors(b)

In [ ]:
%%writefile adders.py
"""Reversible adders built from Feynman, Peres, TSG and R2 gates (Phase 4).

Two 4-bit adders are provided, each as a circuit of Toffoli-type gates:

  cla_adder()   carry look-ahead adder in the style of Kumar et al.,
                "An Optimised Design of a 4-bit Carry Look-Ahead Adder Using
                Novel Reversible Logic Gates", IACIS 2024.
  rca_adder()   ripple-carry adder from two Peres gates per bit.

IMPORTANT: the CLA netlist is a RECONSTRUCTION. The paper gives a block
diagram (its Fig. 6), not a netlist, and the diagram lets one signal feed
several blocks, which a reversible circuit cannot do without copy gates.
The circuit here follows the paper's building blocks and equations:

    PGA block   Peres(A_i, B_i, 0) gives P_i = A_i xor B_i and G_i = A_i B_i;
                Feynman(P_i, C_i) gives S_i = P_i xor C_i
    C1 = G0 xor P0 C0                               Peres + Feynman
    C2 = G1 xor P1 G0 xor P1 P0 C0                  Peres gates + TSG
    C3 = G2 xor P2 G1 xor P2 P1 G0 xor P2 P1 P0 C0  Peres gates + R2

and adds a Feynman copy gate wherever a signal is needed twice.
"""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Dict, List, Sequence, Tuple

from revcircuit import CNOT, Circuit, G, Gate, TOF

# Quantum cost of each block as stated in the adder paper.
PAPER_BLOCK_COST = {"Feynman": 1, "Peres": 4, "TSG": 17, "R2": 5}

# Figures the adder paper reports for its own design (its Table I).
PAPER_REPORTED = {"gates": 20, "ancilla": 14, "garbage": 14, "cost": 70, "trlic": 118}


# ---------------------------------------------------------------------------
# Gate library: each block as Toffoli-type gates, and its defining function
# ---------------------------------------------------------------------------
def feynman(a: int, b: int) -> List[Gate]:
    """Feynman gate: (A, B) -> (A, A xor B)."""
    return [CNOT(a, b)]


def peres(a: int, b: int, c: int) -> List[Gate]:
    """Peres gate: (A, B, C) -> (A, A xor B, AB xor C)."""
    return [TOF(a, b, c), CNOT(a, b)]


def tsg(a: int, b: int, c: int, d: int) -> List[Gate]:
    """TSG gate of Thapliyal and Srinivas: (A, B, C, D) -> (A, Q, Q xor D, QD xor AB xor C)
    with Q = A'C' xor B'. The outputs appear on lines a, b, d, c in that order."""
    return [TOF(a, b, c), CNOT(a, b), G([c], [a], b), TOF(b, d, c), CNOT(b, d)]


def r2(a: int, b: int, c: int, d: int) -> List[Gate]:
    """R2 gate: (A, B, C, D) -> (A, B, C, D xor A xor B xor C)."""
    return [CNOT(a, d), CNOT(b, d), CNOT(c, d)]


def feynman_ref(a, b):
    return a, a ^ b


def peres_ref(a, b, c):
    return a, a ^ b, (a & b) ^ c


def tsg_ref(a, b, c, d):
    q = ((1 - a) & (1 - c)) ^ (1 - b)
    return a, q, q ^ d, (q & d) ^ (a & b) ^ c


def r2_ref(a, b, c, d):
    return a, b, c, d ^ a ^ b ^ c


# ---------------------------------------------------------------------------
# A design: circuit plus the meaning of its lines
# ---------------------------------------------------------------------------
@dataclass
class Design:
    name: str
    circuit: Circuit
    line_names: List[str]
    inputs: Dict[str, int]                 # primary input name -> line
    ancilla: List[int]                     # lines that must start at 0
    outputs: Dict[str, int]                # primary output name -> line
    blocks: List[Tuple[str, Tuple[int, ...]]] = field(default_factory=list)

    @property
    def n(self) -> int:
        return self.circuit.n

    @property
    def garbage(self) -> List[int]:
        """Every output line that is not a primary output."""
        used = set(self.outputs.values())
        return [l for l in range(self.n) if l not in used]

    def block_count(self) -> int:
        return len(self.blocks)

    def block_cost(self) -> int:
        """Quantum cost with the block costs stated in the adder paper."""
        return sum(PAPER_BLOCK_COST[kind] for kind, _ in self.blocks)

    def block_summary(self) -> str:
        kinds = [kind for kind, _ in self.blocks]
        return ", ".join(f"{kinds.count(k)} {k}" for k in ("Peres", "Feynman", "TSG", "R2")
                         if k in kinds)

    def with_circuit(self, circuit: Circuit, name: str) -> "Design":
        """The same interface around another circuit (e.g. an optimized one)."""
        return Design(name, circuit, self.line_names, self.inputs, self.ancilla,
                      self.outputs, [])

    def run(self, values: Dict[str, int]) -> Dict[str, int]:
        """Simulate with the given primary input bits; ancilla lines start at 0."""
        state = sum(values[name] << line for name, line in self.inputs.items())
        out = self.circuit.apply(state)
        return {name: (out >> line) & 1 for name, line in self.outputs.items()}


class _Builder:
    def __init__(self, names: Sequence[str]) -> None:
        self.names = list(names)
        self.index = {name: i for i, name in enumerate(self.names)}
        self.gates: List[Gate] = []
        self.blocks: List[Tuple[str, Tuple[int, ...]]] = []

    def add(self, kind: str, *line_names: str) -> None:
        lines = tuple(self.index[name] for name in line_names)
        maker = {"Feynman": feynman, "Peres": peres, "TSG": tsg, "R2": r2}[kind]
        self.gates += maker(*lines)
        self.blocks.append((kind, lines))


# ---------------------------------------------------------------------------
# Carry look-ahead adder (reconstruction of Kumar et al.)
# ---------------------------------------------------------------------------
def cla_adder(carry_out: bool = False) -> Design:
    """4-bit reversible carry look-ahead adder.

    carry_out=False  the paper's block inventory: sum bits S0..S3 only.
    carry_out=True   the last Feynman gate is replaced by a Peres gate that
                     also produces the final carry C4 = G3 xor P3 C3.
    """
    names = ([f"a{i}" for i in range(4)] + [f"b{i}" for i in range(4)] + ["c0"]
             + [f"g{i}" for i in range(4)]            # G_i
             + ["k0", "k1", "k2", "k3"]               # copies
             + [f"t{i}" for i in range(1, 7)]         # product terms
             + ["z"])                                 # constant input of the TSG gate
    b = _Builder(names)
    for i in range(4):                                # PGA, Peres part: b_i = P_i, g_i = G_i
        b.add("Peres", f"a{i}", f"b{i}", f"g{i}")
    b.add("Feynman", "c0", "k0")                      # copy C0
    b.add("Feynman", "b0", "c0")                      # PGA 0, Feynman part: c0 = S0
    b.add("Peres", "b0", "k0", "t1")                  # t1 = P0 C0
    b.add("Feynman", "g0", "k1")                      # copy G0
    b.add("Feynman", "t1", "k1")                      # k1 = C1 = G0 xor P0 C0
    b.add("Peres", "b1", "g0", "t2")                  # t2 = P1 G0
    b.add("Peres", "b1", "t1", "t3")                  # t3 = P1 P0 C0
    b.add("Feynman", "b1", "k1")                      # PGA 1, Feynman part: k1 = S1
    b.add("Feynman", "g1", "k2")                      # copy G1
    b.add("Feynman", "t2", "k3")                      # copy P1 G0
    b.add("TSG", "t3", "k3", "z", "k2")               # k2 = C2 (three-input XOR)
    b.add("Peres", "b2", "g1", "t4")                  # t4 = P2 G1
    b.add("Peres", "b2", "t2", "t5")                  # t5 = P2 P1 G0
    b.add("Peres", "b2", "t3", "t6")                  # t6 = P2 P1 P0 C0
    b.add("Feynman", "b2", "k2")                      # PGA 2, Feynman part: k2 = S2
    b.add("R2", "t4", "t5", "t6", "g2")               # g2 = C3 (four-input XOR)
    outputs = {"s0": "c0", "s1": "k1", "s2": "k2", "s3": "g2"}
    if carry_out:
        b.add("Peres", "b3", "g2", "g3")              # g2 = S3, g3 = C4
        outputs["c4"] = "g3"
    else:
        b.add("Feynman", "b3", "g2")                  # PGA 3, Feynman part: g2 = S3
    ix = b.index
    return Design(
        name="CLA with carry-out" if carry_out else "CLA (paper style)",
        circuit=Circuit(len(names), b.gates), line_names=names,
        inputs={n: ix[n] for n in names[:9]},
        ancilla=[ix[n] for n in names[9:]],
        outputs={k: ix[v] for k, v in outputs.items()}, blocks=b.blocks)


# ---------------------------------------------------------------------------
# Ripple-carry adder: two Peres gates per bit
# ---------------------------------------------------------------------------
def rca_adder() -> Design:
    """4-bit reversible ripple-carry adder with carry-out.

    Per bit: Peres(A_i, B_i, 0) gives P_i and G_i, then Peres(P_i, C_i, G_i)
    gives S_i and C_(i+1) = G_i xor P_i C_i.
    """
    names = ([f"a{i}" for i in range(4)] + [f"b{i}" for i in range(4)] + ["c0"]
             + [f"z{i}" for i in range(4)])
    b = _Builder(names)
    carry = "c0"
    outputs = {}
    for i in range(4):
        b.add("Peres", f"a{i}", f"b{i}", f"z{i}")
        b.add("Peres", f"b{i}", carry, f"z{i}")
        outputs[f"s{i}"] = carry
        carry = f"z{i}"
    outputs["c4"] = carry
    ix = b.index
    return Design(
        name="Ripple-carry (Peres)", circuit=Circuit(len(names), b.gates), line_names=names,
        inputs={n: ix[n] for n in names[:9]}, ancilla=[ix[n] for n in names[9:]],
        outputs={k: ix[v] for k, v in outputs.items()}, blocks=b.blocks)


# ---------------------------------------------------------------------------
# Checks and measures
# ---------------------------------------------------------------------------
def check_adder(design: Design) -> Tuple[int, int]:
    """Test all 512 input combinations. Returns (cases tested, cases wrong)."""
    wrong = 0
    for a in range(16):
        for b in range(16):
            for c0 in range(2):
                values = {f"a{i}": (a >> i) & 1 for i in range(4)}
                values.update({f"b{i}": (b >> i) & 1 for i in range(4)})
                values["c0"] = c0
                out = design.run(values)
                total = a + b + c0
                good = all(out[f"s{i}"] == (total >> i) & 1 for i in range(4))
                if "c4" in out:
                    good = good and out["c4"] == (total >> 4) & 1
                wrong += not good
    return 512, wrong


def add(design: Design, a: int, b: int, c0: int = 0):
    """Add two 4-bit numbers with a design. Returns (sum, carry); carry is None
    when the design has no carry output."""
    values = {f"a{i}": (a >> i) & 1 for i in range(4)}
    values.update({f"b{i}": (b >> i) & 1 for i in range(4)})
    values["c0"] = c0
    out = design.run(values)
    total = sum(out[f"s{i}"] << i for i in range(4))
    return total, out.get("c4")


def depth(circuit: Circuit) -> int:
    """Number of time steps when gates on different lines run in parallel."""
    level = [0] * circuit.n
    for g in circuit:
        step = 1 + max(level[l] for l in g.lines)
        for l in g.lines:
            level[l] = step
    return max(level, default=0)

## Step 2: The simplifier

| Step | What it does |
|---|---|
| constants | Removes a gate that can never fire, and drops a control that is always satisfied |
| garbage | Removes a gate whose result is neither a primary output nor read by a gate that matters |
| alias | Moves a control to another line holding the same value (for example the source of a copy), so the copy becomes garbage |
| xor | Replaces a CNOT whose control is the XOR of two or three other lines by CNOTs from those lines |
| rules | Runs the Phase 3 optimizer in between |

"Never" and "always" refer to the valid inputs only: every combination of the primary inputs, with ancilla lines at 0. All of them are simulated at once.

In [ ]:
%%writefile simplify.py
"""Ancilla- and garbage-aware simplification (Phase 5).

The transformation rules of Feng and Li keep the function of a circuit the
same on EVERY line and for EVERY input. A practical circuit needs less:

  * ancilla lines always start at 0, and
  * garbage outputs are never read.

Two circuits are called design-equivalent here when they give the same
primary outputs for every value of the primary inputs, with all ancilla
lines starting at 0. The steps below keep design-equivalence. They are NOT
among the rules of the paper, whose authors leave the case with ancillary
bits open; they are heuristics of this project.

  constants  A gate that can never fire is removed, and a control that is
             always satisfied is dropped. "Never" and "always" refer to the
             valid inputs only (ancilla lines at 0).
  garbage    A gate is removed when its target is not a primary output and
             is not read later by any gate that matters.
  alias      A control is moved to another line that holds the same value at
             that point (for example the source of a copy), after commuting
             the gate earlier if needed. Copy gates then become garbage.
  xor        A CNOT whose control equals the XOR of two or three other lines
             is replaced by CNOTs from those lines, when that makes the gates
             that built the control garbage.
  rules      The Phase 3 optimizer (the paper's rules) runs in between.

The valid inputs are simulated all at once: each line holds one integer with
one bit per combination of the primary inputs. Every step is accepted only
if it lowers (cost, gate count, lines used), and with check=True the primary
outputs are compared with the original on all valid inputs after each step.
"""
from __future__ import annotations

from collections import Counter
from dataclasses import dataclass, field
from itertools import combinations
from typing import Callable, Iterable, List, Sequence, Tuple

from revcircuit import CNOT, Circuit, G, Gate
from optimizer import circuit_cost, commute_tag, optimize, quantum_cost
from bitsim import input_vectors

ALL_STEPS = ("constants", "garbage", "alias", "xor", "rules")


@dataclass
class SimplifyResult:
    original: Circuit
    circuit: Circuit
    inputs: List[int]
    ancilla: List[int]
    outputs: List[int]
    step_counts: Counter = field(default_factory=Counter)
    cost_fn: Callable[[Gate], int] = quantum_cost

    def _used(self, circuit: Circuit) -> set:
        touched = set()
        for g in circuit:
            touched |= g.lines
        return touched

    def ancilla_used(self, circuit: Circuit = None) -> int:
        """Ancilla lines that some gate still touches."""
        circuit = self.circuit if circuit is None else circuit
        return len(self._used(circuit) & set(self.ancilla))

    def lines_used(self, circuit: Circuit = None) -> int:
        circuit = self.circuit if circuit is None else circuit
        return len(self.inputs) + self.ancilla_used(circuit)

    def garbage(self, circuit: Circuit = None) -> int:
        """Used lines that are not primary outputs."""
        return self.lines_used(circuit) - len(self.outputs)

    def summary(self) -> str:
        def change(before, after):
            pct = 100.0 * (before - after) / before if before else 0.0
            return f"{before} -> {after} ({pct:.1f}% lower)"

        o, c = self.original, self.circuit
        lines = [
            f"Gate count:     {change(len(o), len(c))}",
            f"Quantum cost:   {change(circuit_cost(o, self.cost_fn), circuit_cost(c, self.cost_fn))}",
            f"Ancilla inputs: {change(self.ancilla_used(o), self.ancilla_used())}",
            f"Garbage lines:  {change(self.garbage(o), self.garbage())}",
        ]
        if self.step_counts:
            lines.append("Steps applied: " + ", ".join(
                f"{name} x{count}" for name, count in sorted(self.step_counts.items())))
        return "\n".join(lines)


class _Care:
    """Simulation over the valid inputs, and the simplification steps."""

    def __init__(self, n: int, inputs: Sequence[int], ancilla: Sequence[int],
                 outputs: Sequence[int], cost: Callable[[Gate], int]) -> None:
        if sorted(list(inputs) + list(ancilla)) != list(range(n)):
            raise ValueError("every line must be either a primary input or an ancilla")
        self.n, self.outputs, self.cost = n, list(outputs), cost
        self.mask = (1 << (1 << len(inputs))) - 1
        self.start = [0] * n
        for vector, line in zip(input_vectors(len(inputs)), inputs):
            self.start[line] = vector
        self.ancilla = set(ancilla)
        self.counts: Counter = Counter()

    # -- simulation ---------------------------------------------------------
    def activation(self, g: Gate, v: List[int]) -> int:
        """One bit per valid input: 1 where the gate fires."""
        act = self.mask
        for l in g.pos:
            act &= v[l]
        for l in g.neg:
            act &= ~v[l]
        return act & self.mask

    def states(self, gates: Sequence[Gate]) -> List[List[int]]:
        """Line values before each gate, and after the last one."""
        v = list(self.start)
        out = [list(v)]
        for g in gates:
            v[g.target] ^= self.activation(g, v)
            out.append(list(v))
        return out

    def signature(self, gates: Sequence[Gate]) -> Tuple[int, ...]:
        """The primary outputs for all valid inputs."""
        final = self.states(gates)[-1]
        return tuple(final[l] for l in self.outputs)

    def score(self, gates: Sequence[Gate]) -> Tuple[int, int, int]:
        touched = set()
        for g in gates:
            touched |= g.lines
        return (sum(self.cost(g) for g in gates), len(gates), len(touched & self.ancilla))

    # -- steps --------------------------------------------------------------
    def constants(self, gates: List[Gate]) -> List[Gate]:
        v = list(self.start)
        out = []
        for g in gates:
            act = self.activation(g, v)
            if act == 0:
                self.counts["constants: gate never fires"] += 1
                continue
            reduced = True
            while reduced:
                reduced = False
                for l in sorted(g.controls):
                    smaller = G(g.pos - {l}, g.neg - {l}, g.target)
                    if self.activation(smaller, v) == act and self.cost(smaller) <= self.cost(g):
                        g, reduced = smaller, True
                        self.counts["constants: control dropped"] += 1
                        break
            v[g.target] ^= act
            out.append(g)
        return out

    def garbage(self, gates: List[Gate], count: bool = True) -> List[Gate]:
        live = set(self.outputs)
        kept = []
        for g in reversed(gates):
            if g.target in live:
                kept.append(g)
                live |= g.controls
            elif count:
                self.counts["garbage: gate removed"] += 1
        return kept[::-1]

    def alias(self, gates: List[Gate]) -> Tuple[List[Gate], bool]:
        states = self.states(gates)
        base = self.score(gates)
        for j, g in enumerate(gates):
            i = j
            while True:
                v = states[i]
                for u in sorted(g.controls):
                    if v[u] in (0, self.mask):
                        continue
                    for w in range(self.n):
                        if w in g.lines:
                            continue
                        same = v[w] == v[u]
                        opposite = v[w] == (~v[u] & self.mask)
                        if not (same or opposite):
                            continue
                        positive = (u in g.pos) == same
                        pos = (g.pos - {u}) | ({w} if positive else set())
                        neg = (g.neg - {u}) | (set() if positive else {w})
                        trial = gates[:i] + [G(pos, neg, g.target)] + gates[i:j] + gates[j + 1:]
                        trial = self.garbage(trial, count=False)
                        if self.score(trial) < base:
                            self.counts["alias: control moved to an equal line"] += 1
                            return trial, True
                if i == 0 or commute_tag(g, gates[i - 1]) is None:
                    break
                i -= 1
        return gates, False

    def xor(self, gates: List[Gate]) -> Tuple[List[Gate], bool]:
        states = self.states(gates)
        base = self.score(gates)
        for j, g in enumerate(gates):
            if len(g.pos) != 1 or g.neg:
                continue
            (u,) = g.pos
            v = states[j]
            lines = [l for l in range(self.n) if l not in (u, g.target) and v[l] != 0]
            for size in (2, 3):
                for combo in combinations(lines, size):
                    total = 0
                    for l in combo:
                        total ^= v[l]
                    if total != v[u]:
                        continue
                    trial = gates[:j] + [CNOT(l, g.target) for l in combo] + gates[j + 1:]
                    trial = self.garbage(trial, count=False)
                    if self.score(trial) < base:
                        self.counts["xor: control rebuilt from other lines"] += 1
                        return trial, True
        return gates, False

    def rules(self, gates: List[Gate]) -> List[Gate]:
        result = optimize(Circuit(self.n, gates), cost=self.cost)
        if self.score(result.circuit.gates) < self.score(gates):
            self.counts["rules: Phase 3 rewrites"] += result.rewrites
            return list(result.circuit.gates)
        return gates


def simplify(circuit: Circuit, inputs: Sequence[int], ancilla: Sequence[int],
             outputs: Sequence[int], steps: Iterable[str] = ALL_STEPS,
             cost: Callable[[Gate], int] = quantum_cost, check: bool = True) -> SimplifyResult:
    """Simplify a circuit, using that ancilla lines start at 0 and that only
    the lines in `outputs` are read.

    inputs    lines that carry primary inputs
    ancilla   lines that start at 0
    outputs   lines that carry primary outputs; all other outputs are garbage
    steps     which steps to use, any of ALL_STEPS
    """
    steps = set(steps)
    if not steps <= set(ALL_STEPS):
        raise ValueError(f"steps must come from {ALL_STEPS}")
    care = _Care(circuit.n, inputs, ancilla, outputs, cost)
    gates = list(circuit.gates)
    reference = care.signature(gates)

    def confirm(new: List[Gate]) -> List[Gate]:
        if check and care.signature(new) != reference:
            raise AssertionError("a simplification step changed a primary output")
        return new

    while True:
        before = care.score(gates)
        if "constants" in steps:
            gates = confirm(care.constants(gates))
        if "garbage" in steps:
            gates = confirm(care.garbage(gates))
        for name, step in (("alias", care.alias), ("xor", care.xor)):
            changed = name in steps
            while changed:
                gates, changed = step(gates)
                confirm(gates)
        if "rules" in steps:
            gates = confirm(care.rules(gates))
        if care.score(gates) >= before:
            break
    return SimplifyResult(circuit, Circuit(circuit.n, gates), list(inputs), list(ancilla),
                          list(outputs), care.counts, cost)


def simplify_design(design, steps: Iterable[str] = ALL_STEPS, **kw) -> SimplifyResult:
    """simplify() for a Design from adders.py."""
    return simplify(design.circuit, list(design.inputs.values()), design.ancilla,
                    list(design.outputs.values()), steps, **kw)


# ---------------------------------------------------------------------------
# Counting a circuit the way the adder paper does
# ---------------------------------------------------------------------------
def block_view(circuit: Circuit) -> Tuple[int, int, Counter]:
    """Count blocks and price them as in the adder paper.

    A Toffoli gate next to a CNOT between its two controls is one Peres block
    (cost 4). Every other gate is one block at its own cost: Feynman or NOT 1,
    Toffoli-type gates by the project cost model.
    Returns (number of blocks, cost, how many blocks of each kind).
    """
    def peres_pair(a: Gate, b: Gate) -> bool:
        tof, cnot = (a, b) if len(a.controls) == 2 else (b, a)
        return (len(tof.pos) == 2 and not tof.neg and len(cnot.pos) == 1 and not cnot.neg
                and cnot.pos <= tof.pos and cnot.target in tof.pos)

    kinds: Counter = Counter()
    blocks = cost = i = 0
    gates = circuit.gates
    while i < len(gates):
        if i + 1 < len(gates) and peres_pair(gates[i], gates[i + 1]):
            kinds["Peres"] += 1
            cost += 4
            i += 2
        else:
            g = gates[i]
            c = len(g.controls)
            kinds["NOT" if c == 0 else "Feynman" if c == 1 and g.pos else "Toffoli-type"] += 1
            cost += quantum_cost(g)
            i += 1
        blocks += 1
    return blocks, cost, kinds

## Step 3: Verification script

In [ ]:
%%writefile verify_phase5.py
"""Phase 5 verification: ancilla- and garbage-aware simplification.

  A. Each step on a small hand-made example.
  B. Random circuits: the primary outputs never change. This check uses the
     plain one-input-at-a-time simulator, not the simplifier's own simulation.
  C. The adders before and after, with all 512 input combinations re-tested.
  D. Contribution of each step on the carry look-ahead adder.
  E. The result counted the way the adder paper counts.
  F. The simplified carry look-ahead netlist.

Run:  python verify_phase5.py
"""
from __future__ import annotations

import random
import sys
import time

from revcircuit import CNOT, Circuit, G, TOF, X
from optimizer import circuit_cost
from adders import PAPER_REPORTED, check_adder, cla_adder, depth, rca_adder
from simplify import ALL_STEPS, block_view, simplify, simplify_design

# The circuit drawings use Unicode symbols; make sure they print on Windows too.
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

rng = random.Random(2026)
failures = 0


def report(label: str, ok: bool, detail: str = "") -> None:
    global failures
    failures += not ok
    print(f"  {label:<54}{'PASS' if ok else 'FAIL'}" + (f"  {detail}" if detail else ""))


def primary_outputs(circuit: Circuit, inputs, outputs):
    """Primary outputs for every primary input, by plain simulation (ancilla at 0)."""
    table = []
    for value in range(1 << len(inputs)):
        state = sum(((value >> j) & 1) << line for j, line in enumerate(inputs))
        out = circuit.apply(state)
        table.append(tuple((out >> line) & 1 for line in outputs))
    return table


# ---------------------------------------------------------------------------
# A. Hand-made examples
# ---------------------------------------------------------------------------
def part_a() -> None:
    print("A. Each step on a small example (lines: q0, q1 inputs; the rest ancilla)")
    cases = [
        ("constants: control on a line that is still 0",
         Circuit(4, [TOF(0, 2, 1), CNOT(0, 1)]), [0, 1], [2, 3], [1], ("constants",), 1),
        ("constants: control that is always 1",
         Circuit(4, [X(2), TOF(0, 2, 1)]), [0, 1], [2, 3], [1, 2], ("constants",), 2),
        ("garbage: result that nobody reads",
         Circuit(4, [CNOT(0, 1), TOF(0, 1, 2)]), [0, 1], [2, 3], [1], ("garbage",), 1),
        ("alias: use the original instead of its copy",
         Circuit(4, [CNOT(0, 2), TOF(2, 1, 3)]), [0, 1], [2, 3], [3], ("garbage", "alias"), 1),
        ("xor: skip an intermediate line",
         Circuit(5, [CNOT(0, 2), CNOT(1, 2), CNOT(2, 3)]), [0, 1], [2, 3, 4], [3],
         ("garbage", "xor"), 2),
    ]
    for label, circuit, inputs, ancilla, outputs, steps, expected in cases:
        result = simplify(circuit, inputs, ancilla, outputs, steps)
        same = primary_outputs(circuit, inputs, outputs) == primary_outputs(result.circuit, inputs, outputs)
        report(label, same and len(result.circuit) == expected,
               f"{len(circuit)} -> {len(result.circuit)} gates:  {circuit}   =>   {result.circuit}")
    print()


# ---------------------------------------------------------------------------
# B. Random circuits
# ---------------------------------------------------------------------------
def random_design(n: int):
    gates = []
    for _ in range(rng.randint(8, 30)):
        if rng.random() < 0.8:                                # X, CNOT, Toffoli
            kind = rng.choice((0, 1, 1, 2, 2))
            lines = rng.sample(range(n), kind + 1)
            gates.append(G(lines[1:], [], lines[0]))
        else:                                                 # mixed-polarity gate
            lines = rng.sample(range(n), rng.randint(2, min(4, n)))
            split = rng.randint(0, len(lines) - 1)
            gates.append(G(lines[1:1 + split], lines[1 + split:], lines[0]))
    lines = list(range(n))
    rng.shuffle(lines)
    k = rng.randint(2, n - 1)
    inputs, ancilla = sorted(lines[:k]), sorted(lines[k:])
    outputs = sorted(rng.sample(range(n), rng.randint(1, max(1, n // 2))))
    return Circuit(n, gates), inputs, ancilla, outputs


def part_b() -> None:
    print("B. Random circuits with random ancilla lines and random primary outputs")
    print(f"  {'Lines':<7}{'Circuits':>9}{'Gates before':>14}{'after':>8}"
          f"{'Cost before':>13}{'after':>8}  Result")
    for n, count in ((5, 150), (6, 150), (7, 120), (8, 100), (9, 80)):
        g0 = g1 = c0 = c1 = 0
        ok = True
        for _ in range(count):
            circuit, inputs, ancilla, outputs = random_design(n)
            result = simplify(circuit, inputs, ancilla, outputs)
            ok &= (primary_outputs(circuit, inputs, outputs)
                   == primary_outputs(result.circuit, inputs, outputs))
            ok &= circuit_cost(result.circuit) <= circuit_cost(circuit)
            g0, g1 = g0 + len(circuit), g1 + len(result.circuit)
            c0, c1 = c0 + circuit_cost(circuit), c1 + circuit_cost(result.circuit)
        global failures
        failures += not ok
        print(f"  {n:<7}{count:>9}{g0:>14}{g1:>8}{c0:>13}{c1:>8}  {'PASS' if ok else 'FAIL'}")
    print("  PASS = same primary outputs for every primary input, and no higher cost.\n")


# ---------------------------------------------------------------------------
# C. The adders
# ---------------------------------------------------------------------------
def part_c(designs, results) -> None:
    print("C. Adders before and after simplification (project cost model)")
    print(f"  {'Design':<24}{'Gates':>10}{'Cost':>10}{'Ancilla':>10}{'Garbage':>10}"
          f"{'Depth':>10}{'Cost saved':>12}  Result")
    for d, r in zip(designs, results):
        new = d.with_circuit(r.circuit, d.name)
        cases, wrong = check_adder(new)
        before, after = circuit_cost(d.circuit), circuit_cost(r.circuit)
        ok = wrong == 0 and after <= before
        global failures
        failures += not ok
        print(f"  {d.name:<24}{f'{len(d.circuit)} -> {len(r.circuit)}':>10}"
              f"{f'{before} -> {after}':>10}"
              f"{f'{r.ancilla_used(d.circuit)} -> {r.ancilla_used()}':>10}"
              f"{f'{r.garbage(d.circuit)} -> {r.garbage()}':>10}"
              f"{f'{depth(d.circuit)} -> {depth(r.circuit)}':>10}"
              f"{100 * (before - after) / before:>11.1f}%  {'PASS' if ok else 'FAIL'}")
    print("  PASS = the simplified circuit adds correctly for all 512 input combinations.")
    for d, r in zip(designs, results):
        steps = ", ".join(f"{name} x{count}" for name, count in sorted(r.step_counts.items()))
        print(f"  {d.name}: {steps or 'no step applied'}")
    print()


# ---------------------------------------------------------------------------
# D. Contribution of each step
# ---------------------------------------------------------------------------
def part_d(designs) -> None:
    print("D. Contribution of each step")
    stages = [("no simplification", ()),
              ("rules only (Phase 3)", ("rules",)),
              ("constants", ("constants",)),
              ("constants + garbage", ("constants", "garbage")),
              ("... + alias", ("constants", "garbage", "alias")),
              ("... + xor", ("constants", "garbage", "alias", "xor")),
              ("all steps", ALL_STEPS)]
    for d in designs[:2]:
        print(f"  {d.name}")
        print(f"    {'Steps used':<24}{'Gates':>7}{'Cost':>7}{'Ancilla':>9}{'Garbage':>9}")
        for label, steps in stages:
            r = simplify_design(d, steps)
            ok = check_adder(d.with_circuit(r.circuit, d.name))[1] == 0
            global failures
            failures += not ok
            print(f"    {label:<24}{len(r.circuit):>7}{circuit_cost(r.circuit):>7}"
                  f"{r.ancilla_used():>9}{r.garbage():>9}" + ("" if ok else "  FAIL"))
    print()


# ---------------------------------------------------------------------------
# E. Counted as in the adder paper
# ---------------------------------------------------------------------------
def part_e(designs, results) -> None:
    print("E. Counted the way the adder paper counts (Peres 4, Feynman 1, Toffoli 5)")
    print(f"  {'Design':<34}{'Blocks':>7}{'Ancilla':>9}{'Garbage':>9}{'Cost':>6}{'Total':>7}")
    p = PAPER_REPORTED
    print(f"  {'Reported in the adder paper':<34}{p['gates']:>7}{p['ancilla']:>9}"
          f"{p['garbage']:>9}{p['cost']:>6}{p['trlic']:>7}")
    for d, r in zip(designs, results):
        b0, a0, g0, c0 = d.block_count(), len(d.ancilla), len(d.garbage), d.block_cost()
        print(f"  {d.name + ', as built':<34}{b0:>7}{a0:>9}{g0:>9}{c0:>6}{b0 + a0 + g0 + c0:>7}")
        b1, c1, kinds = block_view(r.circuit)
        a1, g1 = r.ancilla_used(), r.garbage()
        print(f"  {d.name + ', simplified':<34}{b1:>7}{a1:>9}{g1:>9}{c1:>6}{b1 + a1 + g1 + c1:>7}"
              f"   ({', '.join(f'{v} {k}' for k, v in sorted(kinds.items()))})")
    print("  Total = blocks + ancilla + garbage + cost (the paper's TRLIC).")
    print("  Garbage here counts every used line that is not a sum or carry output; the")
    print("  paper does not say how it counts garbage, so its 14 may not be comparable.\n")


# ---------------------------------------------------------------------------
# F. The simplified netlist
# ---------------------------------------------------------------------------
def part_f(design, result) -> None:
    print(f"F. Simplified netlist of '{design.name}'")
    names = design.line_names
    used = sorted(set().union(*(g.lines for g in result.circuit)) | set(design.inputs.values()))
    dropped = [names[l] for l in range(design.n) if l not in used]
    for i, g in enumerate(result.circuit, 1):
        controls = [names[l] for l in sorted(g.pos)] + ["not " + names[l] for l in sorted(g.neg)]
        kind = {0: "NOT", 1: "CNOT", 2: "Toffoli"}.get(len(controls), "MCT")
        print(f"  {i:>2}. {kind:<8}{names[g.target]:<3} ^= {' AND '.join(controls) or '1'}")
    outs = ", ".join(f"{k} on {names[l]}" for k, l in sorted(design.outputs.items()))
    print(f"  Outputs: {outs}")
    print(f"  Ancilla lines no longer needed: {', '.join(dropped) or 'none'}\n")


def main() -> int:
    start = time.time()
    print("Phase 5 verification: ancilla- and garbage-aware simplification\n")
    designs = [cla_adder(), cla_adder(carry_out=True), rca_adder()]
    results = [simplify_design(d) for d in designs]
    part_a()
    part_b()
    part_c(designs, results)
    part_d(designs)
    part_e(designs, results)
    part_f(designs[0], results[0])
    print(f"Time: {time.time() - start:.1f} s")
    print("\nAll checks passed." if failures == 0 else f"\n{failures} check(s) FAILED.")
    return 1 if failures else 0


if __name__ == "__main__":
    sys.exit(main())

## Step 4: Run the verification

Every check should show **PASS**, and the output should end with `All checks passed.`

In [ ]:
!python verify_phase5.py

## Step 5: Try it yourself

The first part simplifies the carry look-ahead adder. Change `steps` to see what each step contributes; the choices are `"constants"`, `"garbage"`, `"alias"`, `"xor"` and `"rules"`.

The second part is a small circuit of your own. You state which lines are primary inputs, which are ancilla (start at 0), and which are primary outputs.

In [ ]:
import importlib
import revcircuit, rules, optimizer, bitsim, adders, simplify
for m in (revcircuit, rules, optimizer, bitsim, adders, simplify):
    importlib.reload(m)

from revcircuit import Circuit, G, X, CNOT, TOF
from adders import cla_adder, rca_adder, check_adder, add
from simplify import simplify, simplify_design, ALL_STEPS

# ---- Part 1: the carry look-ahead adder --------------------------------
design = cla_adder()                      # or cla_adder(carry_out=True), rca_adder()
steps = ALL_STEPS                         # for example ("constants", "garbage")

result = simplify_design(design, steps)
print(design.name)
print(result.summary())

smaller = design.with_circuit(result.circuit, design.name)
cases, wrong = check_adder(smaller)
print(f"Adds correctly in {cases - wrong} of {cases} input combinations")
print("9 + 5 + 1 =", add(smaller, 9, 5, 1)[0])

# ---- Part 2: your own circuit -------------------------------------------
# q0, q1 are inputs; q2, q3 are ancilla; only q3 is read at the end.
c = Circuit(4, [CNOT(0, 2),               # copy q0 to q2
                TOF(2, 1, 3),             # q3 = q0 AND q1, using the copy
                CNOT(1, 2)])              # changes q2, which nobody reads
mine = simplify(c, inputs=[0, 1], ancilla=[2, 3], outputs=[3])
print()
print("Your circuit:", c)
print(c.draw())
print("Simplified:  ", mine.circuit)
print(mine.circuit.draw())
print(mine.summary())